# `scoring_engine.py` — Le moteur qui décide

Les deux modules produisent chacun un score. Ce fichier **tranche**.

La structure est fournie. Les **VALEURS** (seuils) sont à vous : ce sont
elles qui déterminent le comportement du système, et c'est sur elles
que porte l'essentiel de votre justification écrite.

## Imports

On s'appuie sur les deux modules d'analyse :
- `phishing_rules` (module 1) fournit `analyser_phishing()`
- `spam_rules` (module 2) fournit `analyser_spam()`

In [ ]:
import phishing_rules
import spam_rules

---
## SEUILS — À CALIBRER PAR VOUS

Ces valeurs sont un **POINT DE DÉPART**, pas une réponse.
Vous devez les réajuster **APRÈS** avoir mesuré vos résultats (`main.py`),
et documenter dans le rapport le avant/après avec les chiffres.

> Un seuil choisi au hasard puis jamais revu est une **faute de méthode**.

| Constante | Valeur initiale | Rôle |
|---|---|---|
| `SEUIL_PHISHING` | `50` | au-delà : verdict **PHISHING** |
| `SEUIL_PHISHING_SUSPECT` | `25` | au-delà : verdict **SUSPECT** |
| `SEUIL_SPAM` | `40` | au-delà : verdict **SPAM** |
| `SEUIL_SPAM_SUSPECT` | `20` | au-delà : verdict **SUSPECT** |
| `SCORE_MAX` | `100` | borne supérieure d'un score |

In [ ]:
SEUIL_PHISHING = 50        # au-delà : verdict PHISHING
SEUIL_PHISHING_SUSPECT = 25
SEUIL_SPAM = 40            # au-delà : verdict SPAM
SEUIL_SPAM_SUSPECT = 20

SCORE_MAX = 100

---
## `borner(score)`

Ramène un score brut dans l'intervalle **0–100**.

La somme brute des règles dépasse 100 (module 1 : jusqu'à 130).
Deux stratégies possibles :

| Stratégie | Formule | Effet |
|---|---|---|
| **PLAFONNER** | `min(score, 100)` | conserve les petits écarts |
| **NORMALISER** | `score / max_theorique * 100` | écrase tout vers le bas |

Le plafonnement est implémenté ici. Si vous changez de stratégie,
justifiez-le dans le rapport : **ce n'est pas un choix neutre.**

In [ ]:
def borner(score):
    return min(max(int(score), 0), SCORE_MAX)

---
## `niveau_de_risque(score_max)`

Niveau affiché à l'analyste, pour prioriser sa file d'attente.

| Score max | Niveau |
|---|---|
| `>= 75` | Critique |
| `>= 50` | Eleve |
| `>= 25` | Moyen |
| `< 25` | Faible |

In [ ]:
def niveau_de_risque(score_max):
    if score_max >= 75:
        return "Critique"
    if score_max >= 50:
        return "Eleve"
    if score_max >= 25:
        return "Moyen"
    return "Faible"

---
## `ACTIONS`

Action appliquée au courriel selon le verdict rendu.

In [ ]:
ACTIONS = {
    "phishing": "Blocage + alerte analyste",
    "spam": "Dossier indesirables",
    "suspect": "Mise en quarantaine, revue humaine",
    "ham": "Livraison normale",
}

---
## `decider(score_phishing, score_spam)`

Renvoie le verdict : `"phishing"`, `"spam"`, `"suspect"` ou `"ham"`.

**RÈGLE D'ARBITRAGE** — quand les deux scores sont élevés, l'hameçonnage
l'emporte. Les conséquences ne sont pas comparables : une publicité
livrée par erreur agace, un vol d'identifiants compromet
l'organisation. C'est pourquoi on teste le phishing **EN PREMIER**.

In [ ]:
def decider(score_phishing, score_spam):
    if score_phishing >= SEUIL_PHISHING:
        return "phishing"
    if score_spam >= SEUIL_SPAM:
        return "spam"
    if score_phishing >= SEUIL_PHISHING_SUSPECT:
        return "suspect"
    if score_spam >= SEUIL_SPAM_SUSPECT:
        return "suspect"
    return "ham"

---
## `analyser_courriel(courriel, table_marques)`

Chaîne complète pour un courriel :

1. analyse hameçonnage (module 1) et spam (module 2) ;
2. bornage des deux scores bruts ;
3. décision, niveau de risque et action.

Renvoie un dictionnaire de résultat exploitable par `main.py`.

In [ ]:
def analyser_courriel(courriel, table_marques):
    brut_p, details_p = phishing_rules.analyser_phishing(courriel, table_marques)
    brut_s, details_s = spam_rules.analyser_spam(courriel)

    score_p = borner(brut_p)
    score_s = borner(brut_s)
    verdict = decider(score_p, score_s)

    return {
        "courriel": courriel,
        "score_phishing": score_p,
        "score_spam": score_s,
        "details_phishing": details_p,
        "details_spam": details_s,
        "verdict": verdict,
        "niveau": niveau_de_risque(max(score_p, score_s)),
        "action": ACTIONS[verdict],
    }

---
## `afficher_rapport(resultat)`

Rapport lisible pour un courriel, façon console d'analyste.

In [ ]:
def afficher_rapport(resultat):
    c = resultat["courriel"]
    print("=" * 58)
    print("  Email Security Analyzer")
    print("=" * 58)
    print(f"Expediteur : {c.get('name', '')} <{c.get('sender', '')}>")
    print(f"Sujet      : {c.get('subject', '')}")

    print("\n--- Analyse hameconnage ---")
    for points, explication in resultat["details_phishing"]:
        print(f"  +{points:<3} {explication}")
    if not resultat["details_phishing"]:
        print("  (aucun indice)")
    print(f"  Score : {resultat['score_phishing']}/100")

    print("\n--- Analyse spam ---")
    for points, explication in resultat["details_spam"]:
        print(f"  +{points:<3} {explication}")
    if not resultat["details_spam"]:
        print("  (aucun indice)")
    print(f"  Score : {resultat['score_spam']}/100")

    print("\n--- Decision ---")
    print(f"  Verdict          : {resultat['verdict'].upper()}")
    print(f"  Niveau de risque : {resultat['niveau']}")
    print(f"  Action           : {resultat['action']}")
    print("=" * 58)